<a href="https://colab.research.google.com/github/yansi64/EJERCICIOS/blob/EJERCICIOS_GOOGLE_COLAB/Pr%C3%A1ctica04_Botellas_Descartadas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import simpy
import random

# Parámetros del sistema
TIEMPO_MOLDEO = 30        # segundos (media)
TIEMPO_ENFRIAMIENTO = 60  # segundos (constante)
TIEMPO_INSPECCION = 45    # segundos (media)
TAMANO_LOTE_EMPAQUE = 100 # botellas por lote
NUM_MAQUINAS_MOLDEO = 2
NUM_INSPECTORES = 1
PROB_DEFECTO = 0.15        # probabilidad de que una botella sea defectuosa

# Proceso de producción de una botella
def producir_botella(env, id_botella, moldeo, inspeccion, empaque, descartadas):
    # Moldeo
    with moldeo.request() as req:
        yield req
        tiempo = random.normalvariate(TIEMPO_MOLDEO, 5)
        yield env.timeout(tiempo)
        print(f"Botella {id_botella} moldeada en {env.now:.1f}")

    # Enfriamiento
    yield env.timeout(TIEMPO_ENFRIAMIENTO)
    print(f"Botella {id_botella} enfriada en {env.now:.1f}")

    # Inspección
    with inspeccion.request() as req:
        yield req
        tiempo = random.expovariate(1.0/TIEMPO_INSPECCION)
        yield env.timeout(tiempo)

        # Decisión: ¿aprobada o defectuosa?
        if random.random() < PROB_DEFECTO:
            descartadas.append(id_botella)
            print(f"❌ Botella {id_botella} descartada en {env.now:.1f}")
            return
        else:
            print(f"✅ Botella {id_botella} aprobada en {env.now:.1f}")

    # Empaque (se acumulan hasta formar un lote)
    empaque.append(id_botella)
    if len(empaque) >= TAMANO_LOTE_EMPAQUE:
        print(f"📦 Lote de {TAMANO_LOTE_EMPAQUE} botellas empacado en {env.now:.1f}")
        empaque.clear()

# Generador de llegadas de botellas
def llegada_botellas(env, moldeo, inspeccion, empaque, descartadas):
    id_botella = 0
    while True:
        yield env.timeout(random.expovariate(1/20))  # llegada cada ~20 seg
        id_botella += 1
        env.process(producir_botella(env, id_botella, moldeo, inspeccion, empaque, descartadas))

# Configuración del entorno
env = simpy.Environment()
moldeo = simpy.Resource(env, NUM_MAQUINAS_MOLDEO)
inspeccion = simpy.Resource(env, NUM_INSPECTORES)
empaque = []
descartadas = []

# Iniciar simulación
env.process(llegada_botellas(env, moldeo, inspeccion, empaque, descartadas))
env.run(until=500)  # simular 500 segundos

# Resumen final
print("\n--- RESULTADOS ---")
print(f"Botellas descartadas: {len(descartadas)}")
print(f"IDs descartados: {descartadas}")

Botella 1 moldeada en 50.2
Botella 2 moldeada en 63.3
Botella 3 moldeada en 84.4
Botella 4 moldeada en 101.4
Botella 1 enfriada en 110.2
Botella 5 moldeada en 111.7
Botella 2 enfriada en 123.3
Botella 7 moldeada en 138.9
Botella 6 moldeada en 139.1
Botella 3 enfriada en 144.4
Botella 4 enfriada en 161.4
❌ Botella 1 descartada en 171.3
Botella 5 enfriada en 171.7
Botella 8 moldeada en 171.7
✅ Botella 2 aprobada en 197.2
Botella 7 enfriada en 198.9
Botella 6 enfriada en 199.1
✅ Botella 3 aprobada en 211.0
Botella 9 moldeada en 225.9
✅ Botella 4 aprobada en 227.8
Botella 8 enfriada en 231.7
Botella 10 moldeada en 239.0
Botella 11 moldeada en 257.9
Botella 12 moldeada en 260.5
✅ Botella 5 aprobada en 279.7
Botella 9 enfriada en 285.9
Botella 13 moldeada en 286.0
✅ Botella 7 aprobada en 293.8
Botella 14 moldeada en 296.0
Botella 10 enfriada en 299.0
Botella 15 moldeada en 315.9
Botella 11 enfriada en 317.9
Botella 12 enfriada en 320.5
Botella 16 moldeada en 329.3
✅ Botella 6 aprobada en 336